
### Objetivo: 
Determinar os principais fatores responsáveis pelo alto índice de cancelamento na empresa e propor soluções para sua redução.


In [ ]:
#====================================================================================
# Projeto: Python Insights
# Arquivo de entrada: cancelamentos.csv
# Saída: Graficos/*.png
#        Potencial_taxa_cancelamento.xlsx
# Autor: Aidano da Silva Filho
#====================================================================================

import pandas as pd
import plotly.express as px
import itertools
import os
from openpyxl import load_workbook

In [ ]:
# -------------------------------
# 1) Carregamento e tratamento
# -------------------------------
base_dados = pd.read_csv("cancelamentos.csv")

# Removendo a coluna CustomerID (não agrega valor à análise de cancelamento)
base_dados = base_dados.drop(columns="CustomerID")

# Removendo linhas com valores ausentes
base_dados = base_dados.dropna()

In [ ]:
# -------------------------------
# 2) Análise inicial
# -------------------------------

mapa_cancelou = {1: "Sim", 0: "Não"}

cancel_atual_absoluto = base_dados["cancelou"].value_counts().rename(index=mapa_cancelou)
cancel_atual_percent= base_dados["cancelou"].value_counts(normalize=True).rename(index= mapa_cancelou)

cancel_tabela = pd.DataFrame({"Quantidade de cancelamentos": cancel_atual_absoluto,"Percentual de cancelamentos": cancel_atual_percent}).fillna(0)

cancel_tabela["Percentual de cancelamentos"] = cancel_tabela["Percentual de cancelamentos"].astype(float)

display(cancel_tabela.style.format({"Quantidade de cancelamentos": "{:,.0f}","Percentual de cancelamentos": "{:.2%}"}))

In [ ]:
# -------------------------------
# 3) Análise exploratória: geração de gráficos
# -------------------------------
os.makedirs("Graficos", exist_ok=True)


for coluna in base_dados.columns:
    
    if coluna != "cancelou":
        arquivo = f"Graficos/{coluna}.png"
        grafico =px.histogram(base_dados, x= coluna, color="cancelou", text_auto=True)
        grafico.write_image(arquivo)


### 🧠 **Insights**
 1 - **Clientes com contrato mensal apresentam taxa de cancelamento muito elevada**
 
   - Ação: Dar desconto no contrato anual e trimestral.

2 - **Clientes com mais de 4 ligações ao call center apresentam cancelamento muito elevado**

   - Ação: Verificar o problema que não está sendo resolvido;
 
   - Ação: Se o cliente realizar mais de 3 contatos, direcionar para um time especializado.
 
3 - **Atrasos superiores a 20 dias levam o cliente ao cancelamento**
   - Ação: Clientes com 15 dias de atraso terão uma equipe especializada para tratar o caso.


In [ ]:
# -------------------------------
# 4) Simulação de cenários (tabela verdade)
# -------------------------------


# 1. Definir os filtros como colunas booleanas temporárias

base_dados['Filtro_Contrato'] = base_dados["duracao_contrato"] != "Monthly"
base_dados['Filtro_CallCenter'] = base_dados["ligacoes_callcenter"] <= 4
base_dados['Filtro_Atraso'] = base_dados["dias_atraso"] <= 20

filtros_config = [
    ("Contrato_Nao_Mensal", 'Filtro_Contrato'),
    ("CallCenter_<=4", 'Filtro_CallCenter'),
    ("Atraso_<=20dias", 'Filtro_Atraso')
]

# 2. Gerar Tabela Verdade (Todas as combinações de True/False para os 3 filtros)
# True = Aplicar filtro, False = Não aplicar

combinacoes = list(itertools.product([True, False], repeat=3))

lista_resultados = []

for estados in combinacoes:
    
    # Inicia máscara selecionando todo o dataframe
    mask_atual = pd.Series([True] * len(base_dados), index=base_dados.index)
    
    # Aplica os filtros que estão "Ativados" (True) nesta combinação
    for i, ativado in enumerate(estados):
        col_nome, col_mask = filtros_config[i]
        if ativado:
            mask_atual = mask_atual & base_dados[col_mask]
            
    # Filtra os dados
    cenario = base_dados[mask_atual]
    
    # Calcula métricas
    # Se o filtro for muito restritivo e não sobrar ninguém, evita erro de divisão
    taxa = cenario["cancelou"].mean() if len(cenario) > 0 else 0
    
    #qtd_clientes = len(df_cenario)
    
    # Adiciona ao resultado
    lista_resultados.append({
        "Condição Contrato": "ATIVADO" if estados[0] else "DESATIVADO",
        "Condição CallCenter": "ATIVADO" if estados[1] else "DESATIVADO",
        "Condição Atraso": "ATIVADO" if estados[2] else "DESATIVADO",
        "Taxa Cancelamento": taxa
        
    })

# 3. Criar DataFrame com os resultados
analise = pd.DataFrame(lista_resultados)

# Ordenar para mostrar os melhores cenários (menor cancelamento) primeiro
analise = analise.sort_values(by="Taxa Cancelamento", ascending=True).reset_index(drop=True)

arquivo_saida ="Potencial_taxa_cancelamento.xlsx"
# Visualizar resultado
display_resultados = analise.style.format({"Taxa Cancelamento": "{:.2%}"})

analise.to_excel(arquivo_saida, index=False)


In [ ]:

# -------------------------------
# 5) Formatando Excel e mostrando a simulação
# -------------------------------
wb = load_workbook(arquivo_saida)
ws = wb.active

# Encontrando a coluna "Taxa Cancelamento" na linha 1
col_taxa = None

for col in range(1, ws.max_column + 1):
    if ws.cell(row=1, column=col).value == "Taxa Cancelamento":
        col_taxa = col
        break

# Aplicando formato percentual 0.00% na coluna "Taxa Cancelamento" (da linha 2 em diante)

if col_taxa is not None:
    for row in range(2, ws.max_row +1):
        ws.cell(row=row, column=col_taxa).number_format="0.00%"

# Largura da célula baseada no maior conteúdo de texto
max_comprimento_global = 0

for row in ws.iter_rows():
    for cell in row:
         if cell.value is not None:
             tamanho = len(str(cell.value))
             if tamanho > max_comprimento_global:
                 max_comprimento_global = tamanho
                 
largura_final = max_comprimento_global +2

for col in range(1, ws.max_column +1):
    col_letter = ws.cell(row=1, column=col).column_letter
    ws.column_dimensions[col_letter].width = largura_final

wb.save(arquivo_saida)


print("Tabela de Impacto das Combinações de Ações:")
display(display_resultados )

## 🎯Análise de cenários
A análise de cenários indica que a redução mais significativa da taxa de cancelamento ocorre quando há atuação simultânea nos contratos mensais, no alto volume de contatos com o call center e nos atrasos de pagamento superiores a 20 dias.

Caso não seja possível atuar em todas as frentes, a priorização do tratamento de clientes com alto volume de ligações ao call center apresenta o maior impacto isolado na redução do churn.